In [1]:
import time
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn import metrics
from sklearn.metrics.pairwise import rbf_kernel

from goodpoints import compress
from openxai.model import LoadModel
from openxai.dataloader import ReturnLoaders
import sage
import shap
from GroundTruthExperiment import GroundTruthExperiment

_, loader_test = ReturnLoaders(data_name="german", download=False, batch_size=128)
X_test = loader_test.dataset.data
y_test = loader_test.dataset.targets.to_numpy()
model = LoadModel(data_name="german", ml_model="ann", pretrained=True)
model.eval()

ArtificialNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=60, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=100, bias=True)
    (3): ReLU()
    (4): Linear(in_features=100, out_features=2, bias=True)
  )
)

In [2]:
ground_truth = GroundTruthExperiment(
    dataset_name="german",
    X_test=X_test,
    y_test=y_test,
    model=model,n_repeats=3, 
)
ground_truth.perform()

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Running ground truth calculation iteration 1/3
Calculating ground truth for estimator: kernel, explainer: shap
Calculating ground truth for estimator: permutation, explainer: sage


Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Running ground truth calculation iteration 2/3
Calculating ground truth for estimator: kernel, explainer: shap
Calculating ground truth for estimator: permutation, explainer: sage


Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Running ground truth calculation iteration 3/3
Calculating ground truth for estimator: kernel, explainer: shap
Calculating ground truth for estimator: permutation, explainer: sage
Saved aggregated results to metadata/german/ground_truth_0_3.npz for german with 3 repeats.


In [2]:
data = np.load("metadata/german/ground_truth_0_3.npz", allow_pickle=True)
data

NpzFile 'metadata/german/ground_truth_0_3.npz' with keys: dataset_name, method, estimator, explanation, kernel...

In [3]:
for i in data.keys():
    print(i)

dataset_name
method
estimator
explanation
kernel
g
num_bins
seed_compression
seed_explanation
compressed_size
id_compressed
original_size
compression_time
explanation_values
explanation_time


In [4]:
import numpy as np

data = np.load("metadata/german/ground_truth_0_3.npz", allow_pickle=True)

explanations = data["explanation"]
estimators = data["estimator"]
mask = (explanations == 'shap') & (estimators == 'kernel')
shap_values = data["explanation_values"][mask]

explanations = data["explanation"]
estimators = data["estimator"]
mask = (explanations == 'sage') & (estimators == 'permutation')
sage_values = data["explanation_values"][mask]

In [5]:
def metric_mae(x, y):
    return np.mean(np.abs(x-y))

In [6]:
metric_mae(shap_values.mean(axis=0), shap_values[2])

np.float64(0.0017696406233657289)

In [7]:
metric_mae(sage_values.mean(axis=0), sage_values[2])

np.float64(0.0006587309611273063)

In [8]:
from sklearn.metrics import pairwise_distances


def calculate_mmd(X, Y, gamma):
    XX = rbf_kernel(X, X, gamma)
    YY = rbf_kernel(Y, Y, gamma)
    XY = rbf_kernel(X, Y, gamma)
    return XX.mean() + YY.mean() - 2 * XY.mean()


In [9]:
shap_values[2]

array([[-0.02496027, -0.01507609, -0.02445232, ..., -0.06416306,
        -0.05203115,  0.        ],
       [ 0.01747204, -0.00399877,  0.0281467 , ...,  0.00438556,
        -0.0220343 ,  0.00964115],
       [-0.0403239 ,  0.00929766, -0.01906021, ...,  0.00721227,
        -0.02086535,  0.01949678],
       ...,
       [ 0.00379098, -0.0046256 ,  0.03944112, ..., -0.00572674,
        -0.00079443,  0.07645   ],
       [ 0.01790553,  0.00443623,  0.        , ...,  0.00280788,
         0.03077009, -0.02482646],
       [-0.00888119, -0.00264319,  0.        , ...,  0.01592561,
        -0.00417482,  0.0732966 ]])

In [10]:
shap_values.mean(axis=0)

array([[-0.02504431, -0.0140927 , -0.02299045, ..., -0.07920208,
        -0.05190255,  0.00180108],
       [ 0.01731181, -0.00608672,  0.02624387, ...,  0.00463736,
        -0.02413305,  0.00968405],
       [-0.03981972,  0.01024043, -0.02171582, ...,  0.00722089,
        -0.02251903,  0.02063234],
       ...,
       [ 0.00482208, -0.00547343,  0.03438796, ..., -0.00170877,
        -0.00146495,  0.0756176 ],
       [ 0.01806744,  0.00454635, -0.00092303, ...,  0.00445904,
         0.03012902, -0.02442044],
       [-0.00892844, -0.0059963 , -0.00193422, ...,  0.01357042,
        -0.00219049,  0.0711492 ]])

In [11]:
calculate_mmd(sage_values.mean(axis=0).reshape(1, -1), sage_values[2].reshape(1, -1), gamma=0.1)

np.float64(8.407923065334444e-06)

In [12]:
sage_values.mean(axis=0)

array([ 0.00030705, -0.01265703, -0.02085692, -0.00566596, -0.02368647,
       -0.00414182, -0.02640836, -0.00400556, -0.00999332, -0.05478935,
       -0.01780244, -0.00330776, -0.00524338, -0.01897841, -0.00104229,
       -0.0016093 , -0.00343557,  0.0001428 , -0.00715116, -0.04400051,
       -0.0058125 , -0.02023845,  0.00063414,  0.00036586, -0.02327459,
       -0.00685215, -0.00058638, -0.0258387 , -0.0238897 ,  0.00066949,
       -0.00684601,  0.00045739, -0.00951962, -0.00424391, -0.02220197,
       -0.02230032, -0.03477227, -0.00651715,  0.00439105, -0.05128783,
       -0.10143966, -0.01291886, -0.02597521, -0.01931383, -0.01275619,
       -0.03574556, -0.01307057, -0.01782602, -0.03986264, -0.02123206,
       -0.02496354, -0.01814595, -0.02469862, -0.02237874, -0.00740368,
       -0.00508015, -0.02172929, -0.00997384, -0.04335307, -0.03995198])

In [13]:
sage_values.mean(axis=0).reshape(1, -1)

array([[ 0.00030705, -0.01265703, -0.02085692, -0.00566596, -0.02368647,
        -0.00414182, -0.02640836, -0.00400556, -0.00999332, -0.05478935,
        -0.01780244, -0.00330776, -0.00524338, -0.01897841, -0.00104229,
        -0.0016093 , -0.00343557,  0.0001428 , -0.00715116, -0.04400051,
        -0.0058125 , -0.02023845,  0.00063414,  0.00036586, -0.02327459,
        -0.00685215, -0.00058638, -0.0258387 , -0.0238897 ,  0.00066949,
        -0.00684601,  0.00045739, -0.00951962, -0.00424391, -0.02220197,
        -0.02230032, -0.03477227, -0.00651715,  0.00439105, -0.05128783,
        -0.10143966, -0.01291886, -0.02597521, -0.01931383, -0.01275619,
        -0.03574556, -0.01307057, -0.01782602, -0.03986264, -0.02123206,
        -0.02496354, -0.01814595, -0.02469862, -0.02237874, -0.00740368,
        -0.00508015, -0.02172929, -0.00997384, -0.04335307, -0.03995198]])

In [14]:
len(data["explanation_values"])

6

In [15]:
from MetricsCalculator import MetricsCalculator

In [16]:
mc = MetricsCalculator(ground_truth_path="metadata/german/ground_truth_0_3.npz", experiment_path="metadata/german/ground_truth_0_3.npz")

In [18]:
mc.calculate_metrics_save_results()

Results saved to metadata/german/ground_truth_0_3_with_metrics.npz. Metrics were added: MAE, Top-K Accuracy, MMD.


In [23]:
data = np.load("metadata/german/ground_truth_0_3_with_metrics.npz", allow_pickle=True)
data.files

['dataset_name',
 'method',
 'estimator',
 'explanation',
 'kernel',
 'g',
 'num_bins',
 'seed_compression',
 'seed_explanation',
 'compressed_size',
 'id_compressed',
 'original_size',
 'compression_time',
 'explanation_values',
 'explanation_time',
 'mae',
 'top_k',
 'mmd']